# Routing options

When `routing` is requested, SABRE inserts SWAPs so that every two-qubit gate acts on coupled qubits. Every step can be
tuned: the number of trials, the initial layout, the exact-layout search, gate cancellation, bridge gates and the
calibration-aware re-layout. The response reports what the router did.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:a77a42',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. More trials, fewer gates

Each SABRE trial routes the whole circuit; the one with the fewest two-qubit gates is kept. More trials find cheaper circuits and take longer.

In [2]:
prog = scrambler(12, 6)
rows = []
for trials in (1, 2, 4, 16, 32):
    r = run("iqm_emerald", prog, shots=10, routing={"method": "sabre", "trials": trials})
    rows.append({"trials": trials, "SWAPs": r["compilation"]["routing"]["swaps_inserted"],
                 "native 2Q gates": r["compilation"]["native_two_qubit_count"],
                 "router ms": round(r["telemetry"]["breakdown"]["worker_route_ms"], 1)})
pd.DataFrame(rows).set_index("trials")

,SWAPs,native 2Q gates,router ms
trials,,,
1,30,126,9.2
2,23,105,21.6
4,23,105,46.4
16,19,93,199.8
32,17,87,405.7


## 2. Exact layout first

If the circuit's interaction graph fits the coupling graph, a subgraph search places it with no SWAP at all. Switching that search off leaves SABRE to its own heuristics.

In [3]:
rows = []
for name, prog in (("GHZ chain, 12 qubits", ghz(12)), ("QAOA ring, 10 qubits", qaoa_ring(10, 0.4, 0.7))):
    for exact in (True, False):
        r = run("iqm_emerald", prog, shots=10, routing={"method": "sabre", "exact_layout": exact})
        rows.append({"circuit": name, "exact layout": exact, "SWAPs": r["compilation"]["routing"]["swaps_inserted"],
                     "native 2Q gates": r["compilation"]["native_two_qubit_count"]})
pd.DataFrame(rows).pivot(index="circuit", columns="exact layout", values=["SWAPs", "native 2Q gates"])

SWAPs       native 2Q gates      
exact layout         False True            False True 
circuit                                               
GHZ chain, 12 qubits     1     0              12    11
QAOA ring, 10 qubits     0     0              20    20

## 3. Trivial against searched initial layouts

`initial_layout: trivial` keeps logical qubit i on physical qubit i; the default searches for a good start.

In [4]:
rows = []
for prog_name, prog in (("star GHZ 8", star_ghz(8)), ("random 10", scrambler(10, 4))):
    for layout in ("trivial", "sabre"):
        r = run("iqm_emerald", prog, shots=10, routing={"method": "sabre", "initial_layout": layout, "exact_layout": False})
        rows.append({"circuit": prog_name, "initial layout": layout, "SWAPs": r["compilation"]["routing"]["swaps_inserted"],
                     "native 2Q gates": r["compilation"]["native_two_qubit_count"]})
pd.DataFrame(rows).pivot(index="circuit", columns="initial layout", values="native 2Q gates")

initial layout,sabre,trivial
circuit,,
random 10,38,92
star GHZ 8,10,15


## 4. Cancelling gates after routing

After routing, identical two-qubit gates cancel in pairs when everything between them commutes. `optimize: false` keeps them.

In [5]:
prog = star_ghz(10)
for optimize in (True, False):
    r = run("iqm_emerald", prog, shots=10, routing={"method": "sabre", "optimize": optimize})
    rt = r["compilation"]["routing"]
    print(f"optimize={optimize!s:5s}: cancelled {rt['cancelled_two_qubit_gates']:2d}, native 2Q gates {r['compilation']['native_two_qubit_count']}")

optimize=True : cancelled  4, native 2Q gates 14
optimize=False: cancelled  0, native 2Q gates 18


## 5. Bridge gates

A CX between qubits two steps apart can run as a four-CX bridge instead of a SWAP, which leaves the layout unchanged.

In [6]:
prog = star_ghz(10)
for bridge in (False, True):
    r = run("iqm_emerald", prog, shots=10, routing={"method": "sabre", "bridge": bridge})
    rt = r["compilation"]["routing"]
    print(f"bridge={bridge!s:5s}: SWAPs {rt['swaps_inserted']:2d}, bridges {rt['bridges_inserted']:2d}, native 2Q gates {r['compilation']['native_two_qubit_count']}")

bridge=False: SWAPs  3, bridges  0, native 2Q gates 14
bridge=True : SWAPs  0, bridges  5, native 2Q gates 18


## 6. Calibration-aware re-layout

On twins with per-qubit calibration the routed circuit is moved to the best-calibrated qubits with the same couplings. On IBM Heron, plain routing places a seven-qubit GHZ chain on qubit 46, a poor qubit in IBM's snapshot; the re-layout avoids it.

In [7]:
target = ["0" * 7, "1" * 7]
for aware in (False, True):
    r = run("ibm_heron", ghz(7), shots=50000, noise=True, routing={"method": "sabre", "noise_aware": aware})
    used = r["compilation"]["physical_qubits_used"]
    print(f"noise_aware={aware!s:5s}: physical qubits {sorted(used)}, GHZ7 success {success(r, target):.4f}")

noise_aware=False: physical qubits [46, 47, 57, 67, 68, 69, 70], GHZ7 success 0.8404
noise_aware=True : physical qubits [3, 4, 16, 20, 21, 22, 23], GHZ7 success 0.9652


## Takeaways

- Every routing decision is a request option, and every result reports what the router did.
- Exact layout and cancellation come first; trials trade router time for fewer gates.